In [1]:
!pip install -U libsvm-official

  Preparing metadata (setup.py) ... done
  Created wheel for libsvm-official: filename=libsvm_official-3.35.0-cp312-cp312-linux_x86_64.whl size=52532 sha256=2567698dc6cadea795bce66a3b572c86a7926720aadf3cae14b71ca6dd4536bc
  Stored in directory: /home/davide/.cache/pip/wheels/f9/dc/6d/81d5c43e14db8652ef758d9f22c6daf2688e4dcd1a6eb0edc1
Successfully built libsvm-official


In [ ]:
import os
import gzip
import pickle
import numpy as np
import pandas as pd

import seaborn as sns
import matplotlib
matplotlib.use('Agg')  # For saving figures without a GUI
import matplotlib.pyplot as plt

from libsvm.svmutil import svm_train, svm_predict
from sklearn.metrics import confusion_matrix, classification_report

def df_to_libsvm_features(df: pd.DataFrame):
    """
    Convert a DataFrame of shape (N, M) into a list of dictionaries (one per row),
    suitable for libsvm's training/prediction format. Each row is turned into a dict:
       {1: value_of_first_feature, 2: value_of_second_feature, ...}
    """
    arr = df.to_numpy()  # shape (N, M)
    n_rows, n_cols = arr.shape
    data_list = []
    for i in range(n_rows):
        row_dict = {}
        for j in range(n_cols):
            val = arr[i, j]
            if val != 0.0:
                # libsvm typically ignores missing or zero features if not included
                row_dict[j + 1] = float(val)  # +1 so indices start at 1
        data_list.append(row_dict)
    return data_list

def evaluate_top_20(df: pd.DataFrame, race_id_col='race_id', prob_col='pred_prob'):
    """
    For each race_id group, sort by 'prob_col' descending.
    Mark the top 20 rows as 'predicted_top20 = 1', the rest as 0.
    """
    results = []
    for rid, group in df.groupby(race_id_col):
        group = group.sort_values(by=prob_col, ascending=False).copy()
        group['predicted_top20'] = 0
        top_n = min(20, len(group))  # handle races with fewer than 20 cyclists
        group.iloc[:top_n, group.columns.get_loc('predicted_top20')] = 1
        results.append(group)
    return pd.concat(results)

def main():
    print("Loading dataset...")
    with gzip.open('dataset/train_set.pkl', 'rb') as f:
        train_set = pickle.load(f)
    with gzip.open('dataset/test_set.pkl', 'rb') as f:
        test_set = pickle.load(f)
    print("Dataset loaded!")

    # Drop any NaN rows
    train_set = train_set.dropna()
    test_set  = test_set.dropna()

    # Columns we need
    # We expect 'race_id', 'top20', plus our chosen numeric features
    features = ['performance_wma', 'top20_consistency', 'races_dby_consistency']
    print("Features:", features)

    # Create a train_df and test_df with just the columns we care about
    train_df = train_set[['race_id'] + features + ['top20']].copy()
    test_df  = test_set[['race_id'] + features + ['top20']].copy()

    # Separate X and y for the training set
    X_train_df = train_df[features]   # shape (N_train, len(features))
    y_train_df = train_df['top20']    # shape (N_train,)

    # Separate X and y for the test set
    X_test_df = test_df[features]     # shape (N_test, len(features))
    y_test_df = test_df['top20']      # shape (N_test,)

    # Convert to libsvm format
    y_train_list = y_train_df.to_numpy().tolist()     # a list of 0/1 labels
    X_train_list = df_to_libsvm_features(X_train_df)  # list of dicts
    y_test_list  = y_test_df.to_numpy().tolist()
    X_test_list  = df_to_libsvm_features(X_test_df)

    # Train with probability estimates enabled (-b 1)
    # You can add other parameters, e.g. '-s 0 -t 2 -c 1.0 -g 0.05 -b 1'
    # for a C-SVC with RBF kernel, gamma=0.05, etc.
    svm_params = '-b 1'
    print("Training SVM with libsvm...")
    model = svm_train(y_train_list, X_train_list, svm_params)
    print("SVM training complete.")

    # Predict on test with probability
    # Returns (predicted_labels, accuracy, probability_estimates)
    # For binary classification, probability_estimates[i] is [prob_of_class_0, prob_of_class_1]
    p_labels, p_acc, p_probs = svm_predict(y_test_list, X_test_list, model, '-b 1')

    # Usually, libsvm orders classes ascending, e.g. [0, 1], so p_probs[i][1] is the probability for class=1.
    # Check model.label if your classes differ. We'll assume [0, 1].
    pred_prob_class_1 = [row[1] for row in p_probs]

    # Add 'pred_prob' to test_df
    test_df['pred_prob'] = pred_prob_class_1

    # Evaluate top 20 per race
    test_df = evaluate_top_20(test_df)

    # Compare to ground truth
    y_true_test = test_df['top20'].values
    y_pred_test = test_df['predicted_top20'].values

    cm_test = confusion_matrix(y_true_test, y_pred_test)
    cr_test = classification_report(y_true_test, y_pred_test)

    print("\nConfusion Matrix - Test:\n", cm_test)
    print("Classification Report - Test:\n", cr_test)

    # Save results
    os.makedirs('models/libsvm-no-sampling', exist_ok=True)
    plt.figure(figsize=(10,7))
    sns.heatmap(cm_test, annot=True, fmt='d', cmap='Blues')
    plt.title('Confusion Matrix - Test (libsvm, no sampling)')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.savefig('models/libsvm-no-sampling/ConfusionMatrix-Test.png')
    plt.clf()

    with open('models/libsvm-no-sampling/scores-test.txt', 'w') as f:
        f.write(cr_test)

    print("Done. Results saved to 'models/libsvm-no-sampling/'.")

if __name__ == "__main__":
    main()

Loading dataset...
Dataset loaded!
Features: ['performance_wma', 'top20_consistency', 'races_dby_consistency']
Training SVM with libsvm...
...............................................................................................................................................................................